In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/fr-wo-translate/french_wolof_dataset.csv
/kaggle/input/wolofdic/dictionary_fr_wolof.json


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. INSTALLATION DES DÉPENDANCES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip -q install transformers datasets sentencepiece sacrebleu accelerate

In [3]:
!pip -q install evaluate

In [60]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. IMPORTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os
import torch
import numpy as np
import pandas as pd
import evaluate  # pour BLEU

from datasets import load_dataset, DatasetDict
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

print(f"PyTorch version : {torch.__version__}")
print(f"GPU disponible  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PyTorch version : 2.4.0
GPU disponible  : True
GPU             : Tesla T4


In [61]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. CONFIGURATION GLOBALE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Option 1 — MarianMT FR→EN (meilleur point de départ, très proche du Wolof structurellement)
MODEL_NAME = "Helsinki-NLP/opus-mt-fr-en"

# Option 2 — mT5 multilingue (plus lourd mais couvre plus de langues)
#MODEL_NAME = "google/mt5-small"

OUTPUT_DIR    = "./finetuned_fr_wolof"
MAX_LENGTH    = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 30
WEIGHT_DECAY  = 0.01
SEED          = 42

In [62]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4. CHARGEMENT & EXPLORATION DES DONNÉES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ds = load_dataset("galsenai/french-wolof-translation")
print(ds)



DatasetDict({
    train: Dataset({
        features: ['french', 'wolof', 'sources'],
        num_rows: 17777
    })
})


In [63]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4. CHARGEMENT & EXPLORATION DES DONNÉES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#ds = load_dataset("galsenai/french-wolof-translation")
print(ds)

# Aperçu des données — colonnes : "french", "wolof", "sources"
for i, row in enumerate(ds["train"].select(range(3))):
    print(f"\n[Exemple {i+1}]")
    print(f"  FR : {row['french']}")
    print(f"  WO : {row['wolof']}")
    print(f"  Source : {row['sources']}")

DatasetDict({
    train: Dataset({
        features: ['french', 'wolof', 'sources'],
        num_rows: 17777
    })
})

[Exemple 1]
  FR : Cette lettre doit être légalisée par le Ministère des affaires étrangères équatorien et respecter certaines conditions.

  WO : Bataaxal bii jëwriñu lu ajju ci mbiru bitim réew bu Ekuwatër moo ko wara wóoral, te dafa wara mengoo ak yenni càkuteef yi.

  Source : flores

[Exemple 2]
  FR : « On n’a pas eu le temps de se sauver.

  WO : "Amuñu woon benn jot ngir rawal sunu bopp.

  Source : Microsoft Ntrex

[Exemple 3]
  FR : L’Écosse est tout à fait votre maison et nous voulons vraiment que vous restiez ici. »

  WO : Ekost sa dëkk la te bëggna nu nga toog fi."

  Source : Microsoft Ntrex


In [64]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5. STATISTIQUES SUR LES DONNÉES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def compute_stats(split_name):
    split = ds[split_name]
    fr_lens = [len(row["french"].split()) for row in split]
    wo_lens = [len(row["wolof"].split())  for row in split]
    print(f"\n{split_name.upper():>6} | {len(split):>5} exemples")
    print(f"         FR : moy={np.mean(fr_lens):.1f} mots | max={max(fr_lens)} | min={min(fr_lens)}")
    print(f"         WO : moy={np.mean(wo_lens):.1f} mots | max={max(wo_lens)} | min={min(wo_lens)}")

for split in ["train"]:
    compute_stats(split)


 TRAIN | 17777 exemples
         FR : moy=17.2 mots | max=174 | min=1
         WO : moy=17.5 mots | max=807 | min=1


In [65]:
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
def preprocess(examples):
    """Tokenise les paires FR→WO.
    Le dataset a des colonnes directes : 'french', 'wolof', 'sources'.
    """
    inputs  = examples["french"]
    targets = examples["wolof"]

    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False,  # padding dynamique via DataCollator
    )
    return model_inputs


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [66]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5b. SPLIT TRAIN / EVAL / TEST
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Première découpe : 80% train — 20% temp
train_temp = ds["train"].train_test_split(test_size=0.2, seed=SEED)

# Deuxième découpe : le 20% temp → 50% eval / 50% test  (= 10% / 10% du total)
eval_test = train_temp["test"].train_test_split(test_size=0.5, seed=SEED)

ds_splits = DatasetDict({
    "train" : train_temp["train"],   # ~80%
    "eval"  : eval_test["train"],    # ~10%
    "test"  : eval_test["test"],     # ~10%
})

print(ds_splits)
# DatasetDict({
#     train: Dataset(~8000 rows)
#     eval:  Dataset(~1000 rows)
#     test:  Dataset(~1000 rows)
# })

DatasetDict({
    train: Dataset({
        features: ['french', 'wolof', 'sources'],
        num_rows: 14221
    })
    eval: Dataset({
        features: ['french', 'wolof', 'sources'],
        num_rows: 1778
    })
    test: Dataset({
        features: ['french', 'wolof', 'sources'],
        num_rows: 1778
    })
})


In [67]:
tokenized_ds = DatasetDict({
    "train" : ds_splits["train"].map(preprocess, batched=True, remove_columns=["french", "wolof", "sources"]),
    "eval"  : ds_splits["eval"].map(preprocess,  batched=True, remove_columns=["french", "wolof", "sources"]),
    "test"  : ds_splits["test"].map(preprocess,  batched=True, remove_columns=["french", "wolof", "sources"]),
})

Map:   0%|          | 0/14221 [00:00<?, ? examples/s]

Map:   0%|          | 0/1778 [00:00<?, ? examples/s]

Map:   0%|          | 0/1778 [00:00<?, ? examples/s]

In [68]:
print(tokenized_ds)
print("\nTokenisation terminée ✅")

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14221
    })
    eval: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1778
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1778
    })
})

Tokenisation terminée ✅


In [69]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 7. CHARGEMENT DU MODÈLE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
model = MarianMTModel.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modèle         : {MODEL_NAME}")
print(f"Paramètres     : {total_params:,}")
print(f"Entraînables   : {trainable:,}")

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Modèle         : Helsinki-NLP/opus-mt-fr-en
Paramètres     : 75,133,952
Entraînables   : 74,609,664


In [70]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 8. MÉTRIQUE BLEU
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    
    # Remplacer -100 (tokens masqués) par l'id de padding
    if isinstance(preds, tuple):
        preds = preds[0]
    
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Nettoyage
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]  # sacrebleu attend une liste de références
    
    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": round(result["score"], 2)}

In [71]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 9. PARAMÈTRES D'ENTRAÎNEMENT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,  # optimisation pour la mémoire GPU
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Entraînement
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.05,                   # warmup sur 5% des steps
    lr_scheduler_type="cosine",           # scheduler cosine plus stable
    
    # Évaluation & sauvegarde
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,          # recharge le meilleur checkpoint
    metric_for_best_model="bleu",
    greater_is_better=True,
    save_total_limit=2,                   # garde seulement les 2 meilleurs
    
    # Génération pendant l'éval
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    
    # Logs
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",                     # désactiver wandb (mettre "wandb" pour activer)
    
    # Reproductibilité
    seed=SEED,
    fp16=torch.cuda.is_available(),       # mixed precision si GPU dispo
)

print("Paramètres configurés ✅")

Paramètres configurés ✅


/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [72]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 10. INITIALISATION DU TRAINER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["eval"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # stoppe si pas d'amélioration
)

print("Trainer prêt ✅")

Trainer prêt ✅


/tmp/ipykernel_25/1401733424.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [73]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 11. ENTRAÎNEMENT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
train_result = trainer.train()

# Résumé
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
print("✅ Entraînement terminé !")
print(f"   Loss finale    : {metrics.get('train_loss', 'N/A'):.4f}")
print(f"   Steps totaux   : {metrics.get('train_steps_per_second', 'N/A')}")

/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Bleu
1,4.343500,3.989436,0.440000
2,3.508300,3.306372,2.110000
3,3.156300,3.013585,2.790000
4,2.910000,2.826990,3.340000
5,2.715400,2.701724,4.100000
6,2.577500,2.604363,5.370000
7,2.461500,2.524200,5.990000
8,2.381500,2.467258,6.480000
9,2.278500,2.424536,6.760000
10,2.163300,2.385548,7.450000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

***** train metrics *****
  epoch                    =       22.0
  total_flos               =  6173152GF
  train_loss               =     2.3769
  train_runtime            = 3:28:30.76
  train_samples_per_second =     34.101
  train_steps_per_second   =      1.067
✅ Entraînement terminé !
   Loss finale    : 2.3769
   Steps totaux   : 1.067


In [84]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 12. ÉVALUATION SUR LE JEU DE TEST
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
test_results = trainer.evaluate(
    eval_dataset=tokenized_ds["test"],
    metric_key_prefix="test"
)

trainer.log_metrics("test", test_results)
print("📊 Résultats sur le jeu de test :")
for k, v in test_results.items():
    print(f"   {k:<30}: {v:.4f}")

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

***** test metrics *****
  epoch                   =       22.0
  test_bleu               =       9.56
  test_loss               =     2.2645
  test_runtime            = 0:04:40.44
  test_samples_per_second =       6.34
  test_steps_per_second   =        0.2
📊 Résultats sur le jeu de test :
   test_loss                     : 2.2645
   test_bleu                     : 9.5600
   test_runtime                  : 280.4402
   test_samples_per_second       : 6.3400
   test_steps_per_second         : 0.2000
   epoch                         : 22.0000


In [81]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 14. TRADUCTION — EXEMPLES EN INFÉRENCE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def translate(text: str, num_beams: int = 8) -> str:
    """Traduit une phrase du français vers le wolof."""
    model.eval()
    inputs = tokenizer(
        [text],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=4,
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Tests
phrases = [
    "Bonjour, comment vas-tu ?",
    "J'ai faim.",
    "Où vas-tu ?",
    "Je m'appelle Makhtar.",
    "Bonne nuit, dors bien.",
    "Merci beaucoup pour ton aide.",
]

print("=" * 55)
print(f"  {'FRANÇAIS':<30}  {'WOLOF'}")
print("=" * 55)
for phrase in phrases:
    traduction = translate(phrase)
    print(f"  {phrase:<30}  {traduction}")
print("=" * 55)

  FRANÇAIS                        WOLOF
  Bonjour, comment vas-tu ?       Hi, naka la nga?
  J'ai faim.                      Maa ngi xaar.
  Où vas-tu ?                     Fan la nga dem?
  Je m'appelle Makhtar.           Maa ngi tuddu Maxtaar.
  Bonne nuit, dors bien.          Ndax guddi guddi, nelaw bu baax.
  Merci beaucoup pour ton aide.   Jërëjëf lu bari ci sa ndimbal.


In [76]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 13. SAUVEGARDE DU MODÈLE FINAL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Modèle sauvegardé dans : {OUTPUT_DIR} ✅")

# Créer une archive zip pour téléchargement
import shutil
zip_path = shutil.make_archive("finetuned_fr_wolof", "zip", OUTPUT_DIR)
print(f"Archive créée : {zip_path}")

Modèle sauvegardé dans : ./finetuned_fr_wolof ✅
Archive créée : /kaggle/working/finetuned_fr_wolof.zip


In [82]:
print(translate("je suis fatigué"))

sonn naa
